# Combining CHIME and DRAGONS in FD space instead of in frequency space
## A Ordog, July 23, 2026
This is following up on an idea Becky had at the Calgary ISM meeting on July 21.

**One possible pipeline for this, operating on the complex FDF cubes (after Step 1):**

1. Produce DRAGONS FDF in 400-729 MHz range
2. Convolve CHIME FDF to DRAGONS 3.6 degree resolution (low-pass filter CHIME)
3. High-pass filter DRAGONS FDF to remove 30-cm baselines
4. Determine ratio to convert CHIME Jy/beam/RMSF to DRAGONS K/RMSF
    * The RMSF is the same for both given same frequency range
    * Either use a single value for whole map, or declination dependent
5. Apply the ratio to get smoothed CHIME maps in K/RMSF
6. Apply the ratio to get native resolution CHIME maps in K/RMSF
7. Subtract the smooth CHIME maps from native resolution CHIME maps to get residuals
8. Add the residuals to the DRAGONS data from Step 1 (before high-pass filtering)

**A second possible pipeline, which does the conversion to K in the frequency domain:**

1. Convolve CHIME maps to DRAGONS 3.6 degree resolution (low-pass filter CHIME)
2. High-pass filter DRAGONS maps to remove 30-cm baselines
3. Determine ratio to convert CHIME Jy/beam to DRAGONS K
    * Either use a single value for whole map, or declination dependent
4. Apply the ratio to get smoothed CHIME maps in K
5. Apply the ratio to get native resolution CHIME maps in K
6. Produce CHIME FDF from smoothed data in K
7. Produce CHIME FDF from native resolution in K
8. Produce DRAGONS FDF in 400-729 MHz range
9. Subtract the smooth CHIME FDF maps from native resolution CHIME maps to get residuals
10. Add the residuals to the DRAGONS FDF data from Step 8

**Trying the first method for now, because this does not require any new RM synthesis (already done for DRAGONS 400-729 MHz)**

In [ ]:
import numpy as np
from astropy.io import fits as fits
import healpy as hp
from astropy.wcs import WCS
from reproject import reproject_from_healpix, reproject_to_healpix
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import gc

In [ ]:
def convolve_to_beam(map_in_hpx, beam_small, beam_big):

    fwhm = np.sqrt(beam_big**2 - beam_small**2)

    map_in_hpx[np.isnan(map_in_hpx)] = hp.UNSEEN
    
    map_out_hpx = hp.sphtfunc.smoothing(map_in_hpx, fwhm=np.radians(fwhm))

    return map_out_hpx
    

In [ ]:
def FD_moments(data, thresh_map, maxFD, RM_arr, maptype,
                M0=True, M1=True, M2=False):
 
    dFD = abs(RM_arr[1]-RM_arr[0])

    # cut out FD range chosen:
    data_use   = data[(abs(RM_arr) <= maxFD),:,]
    if maptype == 'hpx':
        RM_arr_use = np.expand_dims(RM_arr[(abs(RM_arr) <= maxFD)], axis=(1))
    if maptype == 'plat_car':
        RM_arr_use = np.expand_dims(RM_arr[(abs(RM_arr) <= maxFD)], axis=(1,2))
    print(data_use.shape)
    print(RM_arr_use.shape)

    # set any data points below PI threshold to NaN:
    data_use[data_use < thresh_map] = np.nan

    moments = {}
    if M0:
        print('Calculating M0...')
        M0_data = dFD*np.nansum(data_use, axis=0)
        M0_data[M0_data == 0] = np.nan
        moments['M0'] = M0_data
    if M1:
        print('Calculating M1...')
        M1_data = dFD*np.nansum(data_use*RM_arr_use, axis=0)/M0_data
        moments['M1'] = M1_data
    if M2:
        print('Calculating M2...')
        M2_data = np.sqrt(dFD*np.nansum(data_use*(RM_arr_use - M1_data)**2, axis=0)/M0_data)
        moments['M2'] = M2_data
        
    return(moments)

In [ ]:
chime_hdu   = fits.open('/srv/data/chime/chime_FD_May2024_400_729/FDF_clean_tot.hpx512.fits')
dragons_hdu = fits.open('/srv/data/dragons/chime_compare_400_729/faraday_synthesis/FDF_clean_tot.hpx128.fits')

chime_fdf   = chime_hdu[0].data
dragons_fdf = dragons_hdu[0].data

chime_hdr   = chime_hdu[0].header
dragons_hdr = dragons_hdu[0].header

fd_chime   = WCS(chime_hdr).all_pix2world(0,   range(chime_fdf.shape[0]),  0)[1]
fd_dragons = WCS(dragons_hdr).all_pix2world(0, range(dragons_fdf.shape[0]),0)[1]

In [ ]:
dragons_fdf_us = hp.pixelfunc.ud_grade(dragons_fdf, 512)

print(dragons_fdf_us.shape)
print(chime_fdf.shape)


In [ ]:
i = 400
hp.mollview(chime_fdf[i], min=0, max=5)
hp.mollview(dragons_fdf_us[i], min=0, max=2)


## 2. Convolve CHIME
Note: estimating 15 arcmin now for CHIME beam (I think proper convolution to 400 MHz beam was NOT applied for CHIME previously)

In [ ]:
chime_fdf_conv = np.empty_like(chime_fdf)
    
#for i in range(399,402):
for i in tqdm(range(0,chime_fdf.shape[0])):

    chime_fdf_conv[i] = convolve_to_beam(chime_fdf[i], 15/60, 3.6)


In [ ]:
hp.mollview(chime_fdf[400], min=0, max=5)
hp.mollview(chime_fdf_conv[400], min=0, max=5)

In [ ]:
del dragons_fdf
gc.collect()

## 3. High-pass filter DRAGONS FDF to remove 30-cm baselines
Note: not doing exactly this now, because need to figure out a way to take care of the projection over the whole sky.

## 4. Determine ratio to convert CHIME Jy/beam/RMSF to DRAGONS K/RMSF

In [ ]:
chime_fdf_conv[chime_fdf_conv<-5000] = np.nan

In [ ]:
i = 400

hp.mollview(chime_fdf_conv[i]/dragons_fdf_us[i], cmap='PiYG',min=0,max=2)

In [ ]:
noise_dragons = np.nanmedian(np.concatenate([dragons_fdf_us[0:200,:],dragons_fdf_us[600:800,:]]),axis=0)
noise_chime = np.nanmedian(np.concatenate([chime_fdf_conv[0:200,:],chime_fdf_conv[600:800,:]]),axis=0)


In [ ]:
fig,ax = plt.subplots(1,2,figsize=(12,4))

ax[0].hist(noise_dragons, range=(0,0.1), bins=100);
ax[1].hist(noise_chime,   range=(0,0.1), bins=100);


In [ ]:
chime_conv_M0 = FD_moments(chime_fdf_conv, 6*noise_chime, 50,  fd_chime, 
                              'hpx', M0=True, M1=True, M2=False)['M0']

dragons_M0 = FD_moments(dragons_fdf_us, 6*noise_dragons, 50,  fd_dragons, 
                              'hpx', M0=True, M1=True, M2=False)['M0']

In [ ]:
hp.mollview(chime_conv_M0/dragons_M0, cmap='PiYG',min=0,max=2)

In [ ]:
fig,ax = plt.subplots(1,2,figsize=(12,4))

ax[0].hist(chime_conv_M0/dragons_M0, range=(0,5), bins=100);
ax[0].axvline(x=0.5, color='k')

ax[1].hist(dragons_M0/chime_conv_M0, range=(0,10), bins=100);
ax[1].axvline(x=2, color='k')
#plt.yscale('log')
#for j in range(0,2):
    

## 5. Apply the ratio to get smoothed CHIME maps in K/RMSF

In [ ]:
chime_fdf_conv_K = chime_fdf_conv/0.5


## 6. Apply the ratio to get native resolution CHIME maps in K/RMSF

In [ ]:
chime_fdf_K = chime_fdf/0.5

In [ ]:
print(chime_fdf_conv_K.shape)
print(chime_fdf_K.shape)
hp.mollview(chime_fdf_conv_K[400],min=0,max=2)
hp.mollview(chime_fdf_K[400],min=0,max=2)

In [ ]:
del chime_fdf
del chime_fdf_conv
gc.collect()

## 7. Subtract the smooth CHIME maps from native resolution CHIME maps to get residuals

In [ ]:
residuals = chime_fdf_K - chime_fdf_conv_K
hp.mollview(residuals[400],min=-2,max=2)

In [ ]:
hp.mollview(residuals[400],min=-1,max=1,cmap='RdBu_r')

## 8. Add the residuals to the DRAGONS data from Step 1 (before high-pass filtering)

In [ ]:
chime_dragons_fdf = dragons_fdf_us + residuals

In [ ]:
hp.mollview(dragons_fdf_us[400],min=0,max=1)
hp.mollview(chime_dragons_fdf[400],min=0,max=1)
hp.mollview(chime_fdf_K[400],min=0,max=1)

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(12,6))
ax.hist(chime_fdf_K[400],bins=100,range=(-1,2),alpha=0.5);
ax.hist(dragons_fdf_us[400],bins=100,range=(-1,2),alpha=0.5);
ax.hist(chime_dragons_fdf[400],bins=100,range=(-1,2),alpha=0.5);

In [ ]:
hdr = chime_hdu[0].header
hdr['COMMENT'] = ' combined CHIME and DRAGONS PI in FD domain'
hdr['COMMENT'] = ' AO, July 24, 2026, made with notebook:'
hdr['COMMENT'] = ' gmims-analysis/CHIME_DRAGONS_combine_in_FD.ipynb'
print(repr(hdr))

In [ ]:
fits.writeto('/srv/data/dragons/chime_compare_400_729/faraday_synthesis/FDF_clean_DRAGONS_CHIME.fits', 
             chime_dragons_fdf, hdr, overwrite=False)


In [ ]:
i = 100005
plt.plot(fd_chime, chime_fdf_K[:,i])
plt.plot(fd_chime, dragons_fdf_us[:,i])
plt.plot(fd_chime, chime_dragons_fdf[:,i])
plt.xlim(-50,50)
plt.ylim(0,)

In [ ]:
chime_dragons_fdf.shape